In [ ]:
import numpy as np
import os
from hydra.utils import instantiate
from hydra import initialize, compose
from scipy.linalg import expm
import networkx as nx
from mask_tools import unfaithful_instant_masks, export_lagged_faithful

Masks can be used to enfore specific structures (via booleans masks) or precise variables (with float masks).
Notably, if you want to generate ONLY the masks and no other links, link probabilities should be set to 0.

## Mask for unfaithful relationships. ($V_{faith,l}$,$V_{faith,i}$)

In [ ]:
# Instant. Roughly matches the 0.1 link proba for instant. 
distortion=[0.1, 0.075, 0.05,0.025,0.0]
masks = unfaithful_instant_masks(distortion= distortion,n_unfaithful=1)
for n,x in enumerate(distortion):
    path = "../masks/masks_5_faith_instant_" + str(4-n)+ "/"
    if not os.path.exists(path):
        os.makedirs(path)
    for m,item in enumerate(masks[n]):
        np.save(path + str(m) + ".npy", item)
print(np.abs(masks[masks != 0]).min(), np.abs(masks[masks != 0]).max())
masks = unfaithful_instant_masks(distortion= distortion, n_unfaithful=2,shape=(7, 7))
for n,x in enumerate(distortion):
    path = "../masks/masks_7_faith_instant_" + str(4-n) + "/"
    if not os.path.exists(path):
        os.makedirs(path)
    for m,item in enumerate(masks[n]):
        np.save(path + str(m) + ".npy", item)
print(np.abs(masks[masks != 0]).min(), np.abs(masks[masks != 0]).max())

0.20214057755945777 0.9994091339055327
resampling
resampling
resampling
resampling
resampling
resampling
resampling
resampling
resampling
resampling
resampling
resampling
resampling
resampling
resampling
0.2001811206709518 0.9989998111701204


In [14]:
# Lagged.
distortion=[0.2, 0.1, 0.05,0.025,0.0]
# To have only a single faithful connection per variable  we resample colliding links. This however allows for a maximum of n_vars faithful connections.
# If necessary, fix that in the future.

export_lagged_faithful(distortion=distortion, n_unfaithful=2, shape=(5,5,3))
export_lagged_faithful(distortion=distortion, n_unfaithful=4, shape=(5,5,3) )
export_lagged_faithful(distortion=distortion, n_unfaithful=2, shape=(7,7,4))
export_lagged_faithful(distortion=distortion, n_unfaithful=3, shape=(7,7,4))

Resampling due to divergence. Resamples:  0
resampling due to double links. Count:  20
0.10038021010495746 0.99861137497456
Resampling due to divergence. Resamples:  8
resampling due to double links. Count:  221
0.10061011736773517 0.9999942854345553
Resampling due to divergence. Resamples:  0
resampling due to double links. Count:  8
0.1021425218385349 0.9993216096106216
Resampling due to divergence. Resamples:  0
resampling due to double links. Count:  21
0.10036756592978935 0.9988083297015872


## Additional mask examples

###  Masks for changing coefficients.

In [ ]:
# Example of a boolean mask.
link_proba = 0.075

# Generate 100 for now. 
def make_masks(dim=(5,5,3), link_proba=0.075, path="../masks_5_075/", n=100):
    for x in range(n):
        baseline_mask = np.zeros(dim)
        baseline_mask[np.random.uniform(0,1,baseline_mask.shape) <= link_proba] = 1
        np.save(path + str(x) + ".npy", baseline_mask.astype(bool))
        
if not os.path.exists("../masks/masks_5_075/"):
    os.makedirs("../masks/masks_5_075/")
make_masks(dim=(5,5,3), link_proba=0.075, path="masks/masks_5_075/", n=100)


### Masks for fixed coefficients.


In [ ]:
# Generate 100 for now. 
def make_masks(n_vars,max_lag, link_proba=0.075, path="masks_5_075/", n=100, instant_proba=0.0):
    # We use this directly to check for divergence directly.
    with initialize(version_base=None, config_path="config/"):
        cfg = compose(config_name="ds.yaml")
        
    cfg.generator.struc.link_proba = link_proba
    cfg.n_vars= n_vars
    cfg.generator.struc.max_lags = max_lag
    cfg.generator.instant.link_proba = instant_proba
    generator = instantiate(
        cfg.generator
    )
    for x in range(n):
        X,Y, Y0,_,_,_,_,_  = generator.get_sample()
        if instant_proba > 0.0:
            np.save(path + str(x) + ".npy", Y0[0])
        else:
            np.save(path + str(x) + ".npy", Y[0])


if not os.path.exists("../masks/masks_5_075_exact/"):
    os.makedirs("../masks/masks_5_075_exact/")
make_masks(5,3, link_proba=0.075, path="../masks/masks_5_075_exact/", n=100)

5
